# Retrain DARE2D (leave-one-out)

Companion to `Run_dare2d_Prediction.ipynb`. Where the inference notebook *runs* the pre-trained 8-model
ensemble, this one **retrains** it on the neuroepithelium dataset using the project's
**leave-one-out** protocol: for each set, train on **all sets but that one** and test on it,
producing `checkpoints_set_{n}_all_but_target/best.h5` - exactly the layout `Run_dare2d_Prediction.ipynb` and
the napari plugin consume.

Two stages are trained, as in inference:

1. **Segmentation (center detection)** - a U-Net that finds division centres.
2. **Regression** - estimates division orientation and length.

**Workflow**
1. **Preprocess** each raw set (movie + `division_position*.npy`) into the per-frame
   `previmg / currimg / nextimg / div_location` layout the training generators read.
2. **Train** leave-one-out for segmentation and regression via
   `scripts/batch_train/batch_train_eval.py` (subprocess, streamed here).
3. **Collect** the resulting checkpoints into the `regression_checkpoints/` /
   `segmentation_checkpoints/` layout used by inference.

> A CUDA GPU is strongly recommended - the full ensemble is heavy (2 models x 8 splits x tens
> of epochs). On native-Windows TensorFlow runs on **CPU only**; for GPU training see the
> plugin's `training/` (WSL2 TF-GPU, or the PyTorch backend). See `README.md` for background.

**Run each cell from top to bottom** (Shift+Enter).

---

In [ ]:
import os, sys, subprocess, glob
from pathlib import Path

# Run this notebook from the DARE2d repository root (or set DARE2D_BASE_DIR).
base_dir = Path(os.environ.get("DARE2D_BASE_DIR", Path.cwd())).resolve()

# Raw neuroepithelium sets from Zenodo (record 17442227): each set_N holds one movie
# (.tif/.tiff) plus its division_position*.npy annotation files.
raw_data_dir = base_dir / "data" / "neuroepithelium" / "neuroepithelium"

# Preprocessed sets + leave-one-out runs live here (the data_dir used by the training configs).
prepared_dir = base_dir / "data" / "2d"

assert (base_dir / "scripts" / "batch_train" / "batch_train_eval.py").exists(), f"Run from the DARE2d repo root; batch_train_eval.py not found under {base_dir}"
print("Base dir :", base_dir)
print("Raw sets :", raw_data_dir)
print("Prepared :", prepared_dir)

## 1) Prerequisites

- Install DARE2D in its conda env (see `README.md`: `pip install -r requirements-tf.txt`, then
  `pip install -e .`).
- Download the dataset from Zenodo (record 17442227) and unzip `neuroepithelium.zip` so that
  `data/neuroepithelium/neuroepithelium/set_1 ... set_8` each contain one movie and its
  `division_position*.npy` annotations.
- Run this notebook from the repository root.

## 2) Preprocess the raw sets

The training generators do not read the raw `movie + division_position*.npy`; they read a
per-frame layout (`previmg / currimg / nextimg / div_location`). The core's own preprocessor
`annotator/preprocessing/format_gastru.py` produces it: for each annotated frame `f` it writes
prev = `stack[f-2]`, curr = `stack[f-1]`, next = `stack[f]`, and the `[x, y]` division points.

In [ ]:
# crop_size=0 keeps full frames (the generators crop internally: 64 px for regression,
# 256 px for segmentation). Set >0 to pre-tile each frame into non-overlapping crops.
crop_size = 0
sets = [f"set_{i}" for i in range(1, 9)]

for s in sets:
    src = raw_data_dir / s
    movies = sorted(glob.glob(str(src / "*.tif"))) + sorted(glob.glob(str(src / "*.tiff")))
    annos  = sorted(glob.glob(str(src / "division_position*.npy")))
    if not movies or not annos:
        print(f"!! {s}: missing movie or annotations in {src} - skipping")
        continue
    dst = prepared_dir / s
    dst.mkdir(parents=True, exist_ok=True)
    # format_gastru reads the annotations from --gt_folder and writes prev/curr/next there too.
    for npy in annos:
        tgt = dst / Path(npy).name
        if not tgt.exists():
            tgt.write_bytes(Path(npy).read_bytes())
    cmd = [sys.executable, str(Path("annotator") / "preprocessing" / "format_gastru.py"),
           "--img_path", movies[0], "--gt_folder", str(dst), "--crop_size", str(crop_size)]
    print("\n>", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("\nPreprocessing complete ->", prepared_dir)

## 3) Training hyper-parameters

These match the experiment defaults. Lower `epochs` (e.g. `3`) for a quick smoke test.

In [ ]:
epochs              = 50     # epochs per leave-one-out split
reg_steps_per_epoch = 1000
seg_steps_per_epoch = 250
batch_size          = 32

## 4) Leave-one-out training: regression

`batch_train_eval.py` is launched as a module (`python -m scripts.batch_train.batch_train_eval`)
because it imports from the `scripts` package. It loops over the 8 sets, training each model on
all sets but the target and testing on the target. Interrupt the kernel to stop.

In [ ]:
def run(cmd):
    """Run a subprocess from the repo root, streaming output. Interrupt the kernel to stop."""
    print("> " + " ".join(cmd) + "\n")
    env = dict(os.environ, HYDRA_FULL_ERROR="1")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            bufsize=1, text=True, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"Process exited with code {proc.returncode}")

reg_cmd = [
    sys.executable, "-m", "scripts.batch_train.batch_train_eval",
    "experiment=regression2d",
    "batch_training=regression2d",
    f"epochs={epochs}",
    f"steps_per_epoch={reg_steps_per_epoch}",
    f"batch_size={batch_size}",
]
run(reg_cmd)

## 5) Leave-one-out training: segmentation (center detection)

In [ ]:
seg_cmd = [
    sys.executable, "-m", "scripts.batch_train.batch_train_eval",
    "experiment=segmentation2d",
    "batch_training=center_detection2d",
    f"epochs={epochs}",
    f"steps_per_epoch={seg_steps_per_epoch}",
    f"batch_size={batch_size}",
]
run(seg_cmd)

## 6) Use your retrained ensemble

Each split writes a `checkpoints_set_{n}_all_but_target/best.h5` under that run's Hydra output
directory (shown near the top of each run's log). Gather the regression and segmentation runs
into the two folders inference reads:

```
regression_checkpoints/checkpoints_set_{1..8}_all_but_target/best.h5
segmentation_checkpoints/checkpoints_set_{1..8}_all_but_target/best.h5
```

Then point inference at them:

- **Notebook:** in `Run_dare2d_Prediction.ipynb` set `reg_dir` / `seg_dir` to those two folders.
- **napari plugin:** set the **Regression / Segmentation checkpoint** fields. The plugin's own
  retraining workflow writes runs into `models/<run>/...` and treats the curated `models/best/`
  as read-only - see the plugin's `README.md`.